# LLM retrosynthesis benchmark

Run the end-to-end URSA best-of-N benchmark: sample an LLM, adapt generated routes, score them, and inspect Solv-N metrics.

Before running this notebook:

1. Install dependencies from the repository root: `uv sync --extra llm-benchmark`.
2. Copy `.env.example` to `.env` and add provider credentials.
3. Select the model or Azure deployment in the configuration cell below.

See [`README.md`](README.md) for provider setup, CLI examples, resume behavior, and output descriptions. Secrets stay in `.env`; do not paste API keys into this notebook.

In [ ]:
from pathlib import Path


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the URSA repository")


ROOT = find_repository_root(Path.cwd().resolve())

# LiteLLM model identifier. For Azure, use azure/<deployment-name>.
MODEL = "azure/<deployment-name>"
BENCHMARK = "EXPERT_2026"  # or DRUGS_CLINICALS_2026
OUTPUT = ROOT / "data/results/notebook"

LIMIT = 10  # number of targets; use 0 for all
SAMPLES = 1  # completions per target; use 10 for the full protocol
REQUEST_WORKERS = 8  # concurrent inference requests
SCORE_WORKERS = 0  # ChemCensor processes; use 0 for auto-tuning
FRESH = False  # whether to discard existing completions

ROOT, OUTPUT

## Run the benchmark

The command resumes an existing `completions.jsonl`. Use a separate `OUTPUT` directory for each model. Set `FRESH = True` only when you intentionally want to discard previous completions and start again.

In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    str(ROOT / "scripts/benchmark_llm.py"),
    "--model",
    MODEL,
    "--benchmark",
    BENCHMARK,
    "--samples",
    str(SAMPLES),
    "--request-workers",
    str(REQUEST_WORKERS),
    "--score-workers",
    str(SCORE_WORKERS),
    "--output",
    str(OUTPUT),
]
if LIMIT:
    command.extend(["--limit", str(LIMIT)])
if FRESH:
    command.append("--fresh")

print(" ".join(command))
subprocess.run(command, cwd=ROOT, check=True)

## Inspect results

The metrics include Solv-0/1/2, parser adaptation rate, completion counts, and the number of routes retained for scoring.

In [ ]:
import json

from IPython.display import JSON, display

metrics_path = OUTPUT / "llm_metrics.json"
metrics = json.loads(metrics_path.read_text())
display(JSON(metrics, expanded=True))
metrics